# Sprint 2 Baseline Comparison: SimilarDayNaive vs. Lasso

This notebook reads completed MLflow runs from the `baselines` experiment and renders
the comparison table for Sprint 2. No model logic here — all computation runs via
`scripts/backtest.py`.

**Prerequisite:** both models must have been run with identical `--test-start` / `--test-end`:
```
uv run python scripts/backtest.py --model naive  --test-start 2021-01-01
uv run python scripts/backtest.py --model lasso --test-start 2021-01-01
```

In [ ]:
from pathlib import Path

import mlflow
import pandas as pd

from energy_price_forecast.evaluation.config import EXPERIMENT_NAME

## 1. Load MLflow runs

In [ ]:
mlflow.set_tracking_uri("file:../mlruns")

runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["start_time DESC"],
)

# Keep only the most recent run per run_name.
latest = runs.drop_duplicates(subset="tags.mlflow.runName", keep="first").set_index(
    "tags.mlflow.runName"
)

for name in ("similarday_naive", "lasso"):
    if name not in latest.index:
        raise RuntimeError(
            f"Run '{name}' not found in experiment '{EXPERIMENT_NAME}'. "
            "Run scripts/backtest.py for both models first."
        )

print(f"Naive  run id: {latest.loc['similarday_naive', 'run_id']}")
print(f"Lasso  run id: {latest.loc['lasso', 'run_id']}")

## 2. Guard (E5): same test window and delivery days

In [ ]:
def _param(run_name: str, col: str) -> str:
    return str(latest.loc[run_name, f"params.{col}"])


naive_start = _param("similarday_naive", "test_start")
lasso_start = _param("lasso", "test_start")
naive_end = _param("similarday_naive", "test_end")
lasso_end = _param("lasso", "test_end")

if naive_start != lasso_start or naive_end != lasso_end:
    print(
        f"WARNING: test windows differ! "
        f"naive=({naive_start}, {naive_end}) lasso=({lasso_start}, {lasso_end}). "
        "Re-run both models with identical --test-start / --test-end."
    )
else:
    print(f"Test window: {naive_start} to {naive_end}  [OK]")

In [ ]:
def _load_predictions(run_name: str) -> pd.DataFrame:
    run_id = latest.loc[run_name, "run_id"]
    client = mlflow.tracking.MlflowClient()
    artifacts = client.list_artifacts(run_id)
    parquet_files = [a.path for a in artifacts if a.path.endswith(".parquet")]
    if not parquet_files:
        raise FileNotFoundError(f"No parquet artifact found for run '{run_name}' ({run_id}).")
    local_path = client.download_artifacts(run_id, parquet_files[0])
    return pd.read_parquet(local_path)


preds_naive = _load_predictions("similarday_naive")
preds_lasso = _load_predictions("lasso")

days_naive = set(preds_naive["delivery_day"].unique())
days_lasso = set(preds_lasso["delivery_day"].unique())

if days_naive != days_lasso:
    only_naive = len(days_naive - days_lasso)
    only_lasso = len(days_lasso - days_naive)
    print(
        f"WARNING: delivery day sets differ! "
        f"{only_naive} days only in naive, {only_lasso} days only in lasso."
    )
else:
    print(f"Delivery days: {len(days_naive)} days in both runs  [OK]")

## 3. Comparison table

In [ ]:
METRIC_COLS = [
    "metrics.mae",
    "metrics.rmse",
    "metrics.wape",
    "metrics.mae_per_day_mean",
    "metrics.mae_per_day_std",
    "metrics.mae_per_day_p05",
    "metrics.mae_per_day_p50",
    "metrics.mae_per_day_p95",
]

table = latest.loc[["similarday_naive", "lasso"], METRIC_COLS].copy()
table.columns = [c.replace("metrics.", "") for c in table.columns]
table.index.name = "model"
table = table.astype(float)

# Add relative improvement vs. naive (negative = better).
for col in ("mae", "rmse"):
    baseline_val = table.loc["similarday_naive", col]
    table[f"{col}_vs_naive_%"] = (table[col] - baseline_val) / baseline_val * 100

table.round(3)

## 4. Per-day MAE distribution

In [ ]:
import matplotlib.pyplot as plt


def _per_day_mae(preds: pd.DataFrame) -> pd.Series:
    return (
        preds.dropna(subset=["y_true", "y_pred"])
        .assign(abs_err=lambda df: (df["y_true"] - df["y_pred"]).abs())
        .groupby("delivery_day")["abs_err"]
        .mean()
    )


mae_naive = _per_day_mae(preds_naive)
mae_lasso = _per_day_mae(preds_lasso)

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(
    [mae_naive.values, mae_lasso.values],
    labels=["SimilarDayNaive", "Lasso"],
    patch_artist=True,
)
ax.set_ylabel("Per-day MAE (EUR/MWh)")
ax.set_title("Per-day MAE distribution — Sprint 2 baselines")
plt.tight_layout()
plt.show()

## 5. Interpretation

The **Lasso** is the diagnostic baseline for this project. It captures linear feature
effects while enforcing leakage-free preprocessing (imputation, scaling, and
regularisation fitted on training data only).

The gap between the Lasso and **LightGBM** (Sprint 3) will isolate what the tree
model gains through non-linearity and feature interactions — with full SHAP
attribution to pinpoint which effects drive the improvement.